# ES 667 Deep Learning: Assignment 3
## Small Multimodal Experiments (20 Marks)

### Instructions
- Select `Runtime -> Change runtime type -> T4 GPU` in Google Colab before starting.
- Run all cells sequentially from top to bottom.
- Complete the five functions marked with `YOUR CODE HERE` and fill in the short written explanations.
- Keep all random seeds, model revisions, dataset splits, and training hyperparameters unchanged.
- Submit your completed notebook with all outputs and plots visible.

**Group Members / Roll Numbers:** ...


## Environment and Data Setup
Run the two setup cells below once at the start of your session. The first cell installs `transformers` and `sentence-transformers` and downloads the dataset into `/content/data`. The second cell defines the provided helper utilities for model loading, image preprocessing, plotting, and training.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

PACKAGES = [
    "transformers==5.19.0",
    "sentence-transformers==6.1.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)

DATA_URL = "https://github.com/devansh-lodha/es-667-assignment-3.git"
LOCAL_DATA = Path("data")
COLAB_DATA = Path("/content/data")

if LOCAL_DATA.exists() and (LOCAL_DATA / "clip_vectors.json").exists():
    DATA_DIR = LOCAL_DATA.resolve()
else:
    if not (COLAB_DATA / "clip_vectors.json").exists():
        tmp_repo = Path("/content/_tmp_repo")
        if tmp_repo.exists():
            shutil.rmtree(tmp_repo)
        subprocess.run(["git", "clone", "--depth", "1", DATA_URL, str(tmp_repo)], check=True)
        if COLAB_DATA.exists():
            shutil.rmtree(COLAB_DATA)
        shutil.move(str(tmp_repo / "data"), str(COLAB_DATA))
        shutil.rmtree(tmp_repo)
    DATA_DIR = COLAB_DATA

print("Data directory ready at:", DATA_DIR)


In [ ]:
#@title Provided Helper Functions (Run this cell, do not modify)
from typing import Any, Callable
import gc
import json
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import torch
import torch.nn.functional as F

VLM_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
VLM_REVISION = "7e3e67edbbed1bf9888184d9df282b700a323964"
EMBED_ID = "google/embeddinggemma-2"
EMBED_REVISION = "914f7f89142e33e77833254d9c9b90c3cef7303b"
DIGIT_PROMPT = "What digit is shown? Answer with exactly one digit from 0 to 9."


def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def seed_all(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_gallery() -> tuple[list[dict[str, Any]], list[Image.Image]]:
    manifest_path = DATA_DIR / "gallery" / "manifest.json"
    records = json.loads(manifest_path.read_text())
    for record in records:
        # Strip internal provenance suffix from display titles
        record["title"] = record["title"].split(" · ")[0]
    images = [
        Image.open(DATA_DIR / "gallery" / record["file"]).convert("RGB")
        for record in records
    ]
    return records, images


def show_gallery(
    images: list[Image.Image],
    indices: Any,
    scores: Any = None,
    records: list[dict[str, Any]] | None = None,
) -> None:
    index_list = [int(i) for i in indices]
    fig, axes = plt.subplots(
        1,
        len(index_list),
        figsize=(3 * len(index_list), 3.6),
        squeeze=False,
    )
    for ax, idx in zip(axes.flat, index_list):
        ax.imshow(images[idx])
        ax.axis("off")
        if records is not None:
            title = records[idx].get("title", records[idx].get("id", str(idx)))
        else:
            title = str(idx)
        if scores is not None:
            title += f"\n{float(scores[idx]):.3f}"
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()


def load_embedding_model() -> Any:
    from sentence_transformers import SentenceTransformer

    # EmbeddingGemma 2 requires float32 precision on T4 and CPU.
    return SentenceTransformer(
        EMBED_ID,
        revision=EMBED_REVISION,
        config_kwargs={"audio_config": None},
        model_kwargs={"dtype": torch.float32},
        device=get_device(),
    )


def load_vlm() -> tuple[Any, Any]:
    from transformers import AutoModelForImageTextToText, AutoProcessor

    processor = AutoProcessor.from_pretrained(VLM_ID, revision=VLM_REVISION)
    processor.image_processor.do_image_splitting = False
    model = AutoModelForImageTextToText.from_pretrained(
        VLM_ID,
        revision=VLM_REVISION,
        dtype=torch.float32,
        attn_implementation="sdpa",
    )
    model.to(get_device()).eval()
    return processor, model


def digit_image(array: np.ndarray) -> Image.Image:
    # Preprocess raw 28x28 grayscale array into a 224x224 black-on-white RGB image.
    inverted = ImageOps.invert(Image.fromarray(array.astype("uint8")))
    return inverted.resize((224, 224)).convert("RGB")


def load_digits(split: str) -> tuple[list[Image.Image], np.ndarray]:
    with np.load(DATA_DIR / "digits.npz", allow_pickle=False) as archive:
        raw_images = archive[f"{split}_images"]
        labels = archive[f"{split}_labels"].astype(int)
    images = [digit_image(arr) for arr in raw_images]
    return images, labels


def make_inputs(
    processor: Any,
    model: Any,
    images: list[Image.Image],
    prompt: str = DIGIT_PROMPT,
    answers: Any = None,
) -> dict[str, Any]:
    texts: list[str] = []
    prefix_lengths: list[int] = []

    for idx in range(len(images)):
        messages: list[dict[str, Any]] = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        prefix = processor.apply_chat_template(messages, add_generation_prompt=True)
        if answers is None:
            texts.append(prefix)
        else:
            prefix_batch = processor(
                text=prefix,
                images=[images[idx]],
                return_tensors="pt",
            )
            prefix_lengths.append(prefix_batch["input_ids"].shape[1])
            messages.append(
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": str(answers[idx])}],
                }
            )
            full_text = processor.apply_chat_template(messages, add_generation_prompt=False)
            texts.append(full_text)

    batch = processor(
        text=texts,
        images=[[img] for img in images],
        padding=True,
        return_tensors="pt",
    )

    if answers is not None:
        # Mask user prompt, image tokens, and padding tokens with -100 so loss is computed only on answer tokens.
        labels = batch["input_ids"].clone()
        for idx, prefix_len in enumerate(prefix_lengths):
            labels[idx, :prefix_len] = -100
        labels[batch["attention_mask"] == 0] = -100
        assert (labels != -100).any(dim=1).all(), "No supervised answer tokens found."
        batch["labels"] = labels

    return {
        key: val.to(model.device) if hasattr(val, "to") else val
        for key, val in batch.items()
    }


def parse_digit(text: str) -> int:
    # Remove special chat template tags (such as <end_of_utterance>) before checking for a single digit.
    cleaned = re.sub(r"<[^>]+>", "", text).strip()
    return int(cleaned) if re.fullmatch(r"[0-9]", cleaned) else -1


def evaluate_digits(
    processor: Any,
    model: Any,
    images: list[Image.Image],
    labels: np.ndarray,
    infer_fn: Callable[[Any, Any, Image.Image, str], dict[str, Any]],
) -> dict[str, Any]:
    answers = [infer_fn(processor, model, img, DIGIT_PROMPT)["answer"] for img in images]
    predictions = np.array([parse_digit(ans) for ans in answers])
    return {
        "accuracy": float(np.mean(predictions == labels)),
        "valid_response_fraction": float(np.mean(predictions >= 0)),
        "answers": answers,
        "predictions": predictions.tolist(),
        "labels": labels.tolist(),
    }


def train_connector(
    processor: Any,
    model: Any,
    train_images: list[Image.Image],
    train_labels: np.ndarray,
    val_images: list[Image.Image],
    val_labels: np.ndarray,
    steps: int = 100,
    batch_size: int = 4,
    lr: float = 2e-4,
    seed: int = 42,
) -> dict[str, Any]:
    seed_all(seed)
    for param in model.parameters():
        param.requires_grad_(False)

    trainable_params: list[torch.nn.Parameter] = []
    for name, param in model.named_parameters():
        if ".connector." in name:
            param.requires_grad_(True)
            trainable_params.append(param)

    if not trainable_params:
        raise RuntimeError("No connector parameters found in model.")

    num_trainable = sum(p.numel() for p in trainable_params)
    print("Trainable connector parameters:", num_trainable, flush=True)

    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.0)
    rng = np.random.default_rng(seed)
    history: list[dict[str, float]] = []

    order = rng.permutation(len(train_images))
    cursor = 0

    def compute_validation_loss() -> float:
        model.eval()
        total_loss = 0.0
        total_tokens = 0
        with torch.no_grad():
            for start in range(0, len(val_images), batch_size):
                val_batch = make_inputs(
                    processor,
                    model,
                    val_images[start : start + batch_size],
                    answers=val_labels[start : start + batch_size],
                )
                num_tokens = int((val_batch["labels"][:, 1:] != -100).sum())
                batch_loss = float(model(**val_batch, use_cache=False).loss)
                total_loss += batch_loss * num_tokens
                total_tokens += num_tokens
        return total_loss / total_tokens

    start_time = time.perf_counter()

    # Record initial validation loss at step 0 before any optimizer updates
    initial_val_loss = compute_validation_loss()
    best_val_loss = initial_val_loss
    best_state: dict[str, torch.Tensor] | None = {
        name: param.detach().cpu().clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }
    history.append({"step": 0, "val_loss": initial_val_loss})
    print({"step": 0, "val_loss": round(initial_val_loss, 4), "elapsed_s": round(time.perf_counter() - start_time, 1)}, flush=True)

    for step in range(1, steps + 1):
        if cursor + batch_size > len(order):
            order = rng.permutation(len(train_images))
            cursor = 0
        batch_indices = order[cursor : cursor + batch_size]
        cursor += batch_size

        model.eval()
        batch = make_inputs(
            processor,
            model,
            [train_images[i] for i in batch_indices],
            answers=train_labels[batch_indices],
        )
        optimizer.zero_grad(set_to_none=True)
        loss = model(**batch, use_cache=False).loss
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss at step {step}.")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()

        record: dict[str, float] = {"step": step, "train_loss": float(loss.detach())}
        if step % 25 == 0 or step == steps:
            val_loss = compute_validation_loss()
            record["val_loss"] = val_loss
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {
                    name: param.detach().cpu().clone()
                    for name, param in model.named_parameters()
                    if param.requires_grad
                }
            elapsed = round(time.perf_counter() - start_time, 1)
            print(
                {
                    "step": step,
                    "train_loss": round(record["train_loss"], 4),
                    "val_loss": round(val_loss, 4),
                    "elapsed_s": elapsed,
                },
                flush=True,
            )
        history.append(record)

    if best_state is None:
        raise RuntimeError("No validation checkpoint was recorded.")

    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in best_state:
                param.copy_(best_state[name].to(param.device))

    model.eval()
    return {
        "history": history,
        "seconds": time.perf_counter() - start_time,
        "best_val_loss": best_val_loss,
        "trainable_parameters": num_trainable,
    }


seed_all(42)
print("Active device:", get_device())
print("Pinned revisions:", EMBED_REVISION, VLM_REVISION)


---
## Q1. Contrastive Image-Text Alignment (4 Marks)

In contrastive vision-language models such as CLIP, paired image and text embeddings are aligned in a shared space by maximizing the cosine similarity of matching pairs $(i, i)$ while minimizing the similarity of non-matching pairs $(i, j)$ for $i \neq j$.

You are given $N = 4$ paired image vectors $U \in \mathbb{R}^{N \times D}$ and text vectors $V \in \mathbb{R}^{N \times D}$ in `data/clip_vectors.json`, where row $i$ of $U$ matches row $i$ of $V$.

### Q1(a). Implement Symmetric CLIP Loss (2 Marks)
Complete `clip_symmetric_loss(image_vectors, text_vectors, tau)` using tensor operations and `torch.logsumexp`. Do not use pre-built cross-entropy or contrastive loss functions. Normalize each vector to unit L2 length, construct the temperature-scaled cosine similarity matrix $S_{i,j} = (\hat{u}_i \cdot \hat{v}_j) / \tau$, and compute the symmetric contrastive loss by averaging the mean negative log probability of the diagonal entries across rows and across columns:
$$L = \frac{1}{2} \left( L_{\text{i2t}} + L_{\text{t2i}} \right)$$


In [ ]:
def clip_symmetric_loss(
    image_vectors: torch.Tensor,
    text_vectors: torch.Tensor,
    tau: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute the temperature-scaled similarity matrix and symmetric CLIP loss.

    Args:
        image_vectors: Tensor of shape (N, D) containing unnormalized image vectors.
        text_vectors: Tensor of shape (N, D) containing unnormalized text vectors.
        tau: Positive float temperature parameter.

    Returns:
        scaled_similarity_matrix: Tensor of shape (N, N).
        scalar_loss: 0D scalar Tensor.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Complete Q1(a)")


### Q1(b). Inspect Alignment and Temperature Scaling (1 Mark)
Run the cell below to evaluate your implementation on:
1. Aligned image-text pairs vs permuted text vectors (where the text rows are reordered so diagonal entries no longer correspond to matching pairs).
2. Aligned pairs across three temperatures: $\tau \in \{0.01, 0.1, 1.0\}$.


In [ ]:
clip_data = json.loads((DATA_DIR / "clip_vectors.json").read_text())
image_vectors = torch.tensor(clip_data["image_vectors"], dtype=torch.float32)
text_vectors = torch.tensor(clip_data["text_vectors"], dtype=torch.float32)
permuted_text_vectors = text_vectors[clip_data["permutation"]]

for label, current_texts in [
    ("Aligned pairs", text_vectors),
    ("Permuted text rows", permuted_text_vectors),
]:
    matrix, loss = clip_symmetric_loss(image_vectors, current_texts, clip_data["tau"])
    print(f"=== {label} (tau = {clip_data['tau']}) ===")
    print("Scaled similarity matrix:\n", matrix)
    print("Symmetric loss:", float(loss), "\n")

print("=== Temperature Scaling on Aligned Pairs ===")
for tau_val in [0.01, 0.1, 1.0]:
    _, temp_loss = clip_symmetric_loss(image_vectors, text_vectors, tau_val)
    print(f"tau = {tau_val:4.2f} | Symmetric loss = {float(temp_loss):.6f}")


### Q1(c). Explanation (1 Mark)
In 2 to 3 sentences, explain why the diagonal entries of $S$ represent the target pairs, why permuting only the text rows increases the loss, and how changing the temperature $\tau$ affects the softmax sharpness and resulting loss on aligned pairs.

**Your Explanation:** ...


---
## Q2. Multimodal Image Retrieval with Frozen Embeddings (4 Marks)

In this section, you will build a text-to-image search system over a 17-image gallery using the frozen multimodal model `google/embeddinggemma-2`.

Run the cell below to display the 17 gallery images and compute their L2-normalized image embeddings once. Captions are used only as display titles and are never passed to the embedding model.


In [ ]:
records, gallery_images = load_gallery()
assert len(gallery_images) == 17

for start in range(0, len(records), 5):
    show_gallery(
        gallery_images,
        range(start, min(start + 5, len(records))),
        records=records,
    )

embedding_model = load_embedding_model()
image_embeddings = embedding_model.encode(
    [{"image": image} for image in gallery_images],
    batch_size=1,
    processing_kwargs={"image": {"max_soft_tokens": 280}},
    normalize_embeddings=True,
    convert_to_tensor=True,
    show_progress_bar=True,
)
print("Image index shape:", tuple(image_embeddings.shape))


### Q2(a). Implement Semantic Search (2 Marks)
Complete `semantic_search(query, embedding_model, image_embeddings)` to retrieve the 5 highest-scoring gallery images for a text query without re-encoding the gallery. Encode the query with `embedding_model.encode(query, prompt_name="SearchQuery", convert_to_tensor=True)`, compute cosine similarities against `image_embeddings`, and return the top 5 `(indices, scores)` sorted in descending order.


In [ ]:
def semantic_search(
    query: str,
    embedding_model: Any,
    image_embeddings: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return the top 5 gallery image indices and their cosine similarity scores.

    Args:
        query: Text search string.
        embedding_model: Loaded SentenceTransformer model.
        image_embeddings: Tensor of shape (17, D) containing gallery image embeddings.

    Returns:
        indices: 1D Tensor of shape (5,) containing the top 5 gallery indices.
        scores: 1D Tensor of shape (5,) containing the corresponding cosine similarity scores.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Complete Q2(a)")


### Q2(b). Evaluate Four Queries (1 Mark)
In the code cell below, replace the four empty strings in `queries` with:
1. A straightforward description of an object present in the gallery.
2. A second straightforward description of another object in the gallery.
3. A query targeting a fine-grained visual detail (such as a specific color, clothing item, or visual style).
4. A query for an object completely absent from the 17-image gallery.

Run the cell to display the top 5 retrieved images and cosine similarity scores for each query.


In [ ]:
def display_search(query: str) -> None:
    indices, scores = semantic_search(query, embedding_model, image_embeddings)
    print(f"Query: {query!r}")
    gallery_scores = np.full(len(gallery_images), np.nan)
    for index, score in zip(indices, scores):
        gallery_scores[int(index)] = float(score)
    show_gallery(gallery_images, indices, scores=gallery_scores, records=records)


queries = [
    "",  # 1. Straightforward description 1
    "",  # 2. Straightforward description 2
    "",  # 3. Fine-grained visual detail
    "",  # 4. Object absent from the gallery
]
assert all(q.strip() for q in queries), "Please fill in all four queries."
for q in queries:
    display_search(q)


### Q2(c). Explanation (1 Mark)
In 2 to 3 sentences, discuss the result of your fourth query (the absent item). State what the nearest retrieved images showed and explain why top-$k$ cosine similarity retrieval always returns matches even when the queried concept does not exist in the gallery.

**Your Explanation:** ...


---
## Q3. Vision-Language Model Adaptation (8 Marks)

In this section, you will inspect `HuggingFaceTB/SmolVLM-256M-Instruct` and adapt its vision-to-language connector on a small handwritten digit dataset.

Run the cell below to release the Q2 embedding model from GPU memory and load SmolVLM.


In [ ]:
for var_name in ("embedding_model", "image_embeddings"):
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

processor, vlm = load_vlm()
print("SmolVLM loaded on device:", vlm.device)


### Q3(a). Forward Pass Diagnostics vs Autoregressive Generation (2 Marks)
Complete `infer(processor, model, image, prompt)`:
- Use `make_inputs(processor, model, [image], prompt=prompt)` to tokenize the image and prompt.
- Without gradient tracking, run a single forward pass (`model(**inputs)`) to obtain the output logits, record `logits_shape` as a tuple, and decode the greedy token (`argmax`) from the final sequence position as `next_token`.
- Run deterministic greedy generation (`do_sample=False`, `max_new_tokens=8`), slice off the input prompt tokens so only newly generated tokens remain, and decode them with `skip_special_tokens=True` and `.strip()` as `answer`.


In [ ]:
def infer(
    processor: Any,
    model: Any,
    image: Image.Image,
    prompt: str,
) -> dict[str, Any]:
    """Run a single forward pass for diagnostics, then greedily generate new tokens.

    Args:
        processor: Loaded VLM processor.
        model: Loaded VLM model.
        image: Single PIL Image.
        prompt: Text prompt string.

    Returns:
        Dictionary with keys:
            'answer': str (decoded generated answer with input tokens and special tokens removed)
            'logits_shape': tuple[int, int, int] (batch_size, seq_len, vocab_size)
            'next_token': str (decoded greedy token from the last input position)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Complete Q3(a)")


In [ ]:
photo_indices = [
    i for i, record in enumerate(records) if record["image_kind"] == "photograph"
][:3]
show_gallery(gallery_images, photo_indices, records=records)

for i in photo_indices:
    result = infer(processor, vlm, gallery_images[i], "Describe this photograph briefly.")
    print(
        records[i]["title"],
        "| logits shape:", result["logits_shape"],
        "| next token:", repr(result["next_token"]),
        "| answer:", repr(result["answer"]),
    )


**Explanation (2 sentences):** Explain why a single forward pass (`model(**inputs)`) and generating a multi-token answer (`model.generate(...)`) are different operations, and how `next_token` relates to `answer`.

**Your Explanation:** ...


### Q3(b). Baseline Digit Recognition Before Adaptation (2 Marks)
Run the cell below to evaluate the unadapted VLM on the 100 fixed test digit images using `DIGIT_PROMPT`.
- `accuracy` measures exact match with the ground-truth digit label (0 to 9).
- `valid_response_fraction` measures the fraction of responses that consist of a single digit character without extra words.


In [ ]:
train_images, train_labels = load_digits("train")
val_images, val_labels = load_digits("val")
test_images, test_labels = load_digits("test")
assert (len(train_images), len(val_images), len(test_images)) == (200, 50, 100)

print("Fixed prompt:", DIGIT_PROMPT)
before = evaluate_digits(processor, vlm, test_images, test_labels, infer)
before_first10 = list(before["answers"][:10])
print("Before adaptation:", {k: before[k] for k in ("accuracy", "valid_response_fraction")})


### Q3(c). Connector Fine-Tuning and Post-Adaptation Evaluation (3 Marks)
Run the cell below to fine-tune only the vision-to-language connector (`7,077,888` parameters) while keeping the vision encoder and language model backbones frozen:
- Training set: 200 images, Validation set: 50 images, Seed: 42
- Optimizer: AdamW (`lr=2e-4`, `weight_decay=0.0`, gradient clipping `1.0`), 100 steps, batch size 4
- Validation runs at step 0 and every 25 steps, restoring the checkpoint with the lowest validation answer-token loss.
- After training, the cell re-evaluates the 100 test digits and displays the first 10 test images with before and after predictions.


In [ ]:
training = train_connector(
    processor,
    vlm,
    train_images,
    train_labels,
    val_images,
    val_labels,
    steps=100,
    batch_size=4,
    lr=2e-4,
    seed=42,
)

history = training["history"]
train_rows = [r for r in history if "train_loss" in r]
val_rows = [r for r in history if "val_loss" in r]

plt.figure(figsize=(6, 4))
plt.plot([r["step"] for r in train_rows], [r["train_loss"] for r in train_rows], label="Training")
plt.plot([r["step"] for r in val_rows], [r["val_loss"] for r in val_rows], "o-", label="Validation")
plt.xlabel("Optimizer step")
plt.ylabel("Answer-token loss")
plt.legend()
plt.tight_layout()
plt.show()
print("Restored best validation loss:", training["best_val_loss"])

after = evaluate_digits(processor, vlm, test_images, test_labels, infer)
after_first10 = list(after["answers"][:10])

print("\nStage      Accuracy    Valid-response fraction")
for name, result in [("Before", before), ("After", after)]:
    print(f"{name:10} {result['accuracy']:.3f}       {result['valid_response_fraction']:.3f}")

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i])
    ax.axis("off")
    ax.set_title(
        f"True: {test_labels[i]}\nBefore: {before_first10[i]!r}\nAfter: {after_first10[i]!r}",
        fontsize=9,
        wrap=True,
    )
plt.tight_layout()
plt.show()


### Q3(d). Explanation (1 Mark)
In 3 to 4 sentences, describe how test accuracy and `valid_response_fraction` changed after connector fine-tuning, what types of digit errors were corrected, and whether the baseline errors were caused by output formatting issues or visual recognition errors.

**Your Explanation:** ...


---
## Q4. Concept Bottleneck Routing and Intervention (4 Marks)

In a strict Concept Bottleneck Model (CBM), an image is first mapped to interpretable concepts $c \in \mathbb{R}^C$, and the downstream classifier predicts the final label solely from $c$:
$$\text{image} \longrightarrow c = [\text{is\_red}, \text{is\_circle}] \longrightarrow y \in \{\text{other (0)}, \text{red-circle (1)}\}$$

In `data/cbm.json`, you are given 4 examples with binary concept predictions, along with a linear classification head acting as a logical AND gate:
- Weights: $w = [1.0, 1.0]$
- Bias: $b = -1.5$
- Decision threshold: $\theta = 0.0$


In [ ]:
cbm = json.loads((DATA_DIR / "cbm.json").read_text())
concepts = torch.tensor([r["concept_predictions"] for r in cbm["examples"]], dtype=torch.float32)
weights = torch.tensor(cbm["head"]["weights"], dtype=torch.float32)
bias = float(cbm["head"]["bias"])
threshold = float(cbm["head"]["decision_threshold"])

concept_images = [Image.open(DATA_DIR / r["file"]).convert("RGB") for r in cbm["examples"]]
show_gallery(concept_images, range(len(concept_images)), records=cbm["examples"])
print("Concept names:", cbm["concepts"])
print("Initial concept vectors:\n", concepts)


### Q4(a). Implement the Concept Classification Head (1 Mark)
Complete `cbm_head(concepts, weights, bias, threshold)` to compute the linear score $c \cdot w + b$ and return `(scores, predictions)` where `predictions` is `1` if `scores >= threshold` and `0` otherwise. Your implementation should work for both a 2D batch of shape `(N, 2)` and a 1D vector of shape `(2,)`.


In [ ]:
def cbm_head(
    concepts: torch.Tensor,
    weights: torch.Tensor,
    bias: float,
    threshold: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute linear class scores and binary predictions from concept inputs only.

    Args:
        concepts: Tensor of shape (..., 2) containing [is_red, is_circle].
        weights: Tensor of shape (2,) containing head weights.
        bias: Float scalar bias.
        threshold: Float decision threshold.

    Returns:
        scores: Tensor of shape (...) containing linear scores.
        predictions: Int64 Tensor of shape (...) containing binary predictions (0 or 1).
    """
    # YOUR CODE HERE
    raise NotImplementedError("Complete Q4(a)")


scores, predictions = cbm_head(concepts, weights, bias, threshold)
print("All example scores:", scores, "| all predictions:", predictions)


### Q4(b). Implement Test-Time Concept Intervention (2 Marks)
For the first example (`red_circle`), the initial concept vector is `[0.0, 1.0]` (`is_red` is misclassified as `0`), causing the head to predict `0` (`other`). Complete `intervene(concept_vector, true_is_red)` to return an independent copy of `concept_vector` with only `is_red` (index `0`) updated to `true_is_red`.


In [ ]:
def intervene(
    concept_vector: torch.Tensor,
    true_is_red: float | int,
) -> torch.Tensor:
    """Return an independent copy of concept_vector with index 0 set to true_is_red.

    Args:
        concept_vector: 1D Tensor of shape (2,) representing [is_red, is_circle].
        true_is_red: Ground-truth value for the is_red concept.

    Returns:
        corrected: 1D Tensor of shape (2,).
    """
    # YOUR CODE HERE
    raise NotImplementedError("Complete Q4(b)")


original = concepts[0].clone()
corrected = intervene(original, cbm["examples"][0]["concept_labels"][0])
for label, vector in [("Before intervention", original), ("After intervention", corrected)]:
    score, prediction = cbm_head(vector, weights, bias, threshold)
    print(f"{label:19s} | concepts: {vector.tolist()} | score: {float(score):5.2f} | prediction: {int(prediction)}")


### Q4(c). Explanation (1 Mark)
In 2 to 3 sentences, explain why intervening on `is_red` directly changes the downstream prediction in a strict Concept Bottleneck Model, and why editing a separately generated post-hoc explanation would not change the output of a standard end-to-end black-box classifier.

**Your Explanation:** ...
